In [2]:
"""
Simple Random Forest Example
=============================
Random Forest for Titanic Survival Prediction
"""

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42

# ---------------------------------------------------------------------
# 1. Load the Titanic dataset
# ---------------------------------------------------------------------

df = pd.read_csv("/content/tested.csv")

print("Sample of the dataset:")
print(df.head())

print(f"\nDataset shape: {df.shape}")
print(f"\nClass balance:")
print(df["Survived"].value_counts())

TARGET = "Survived"

# ---------------------------------------------------------------------
# 2. Handle missing values
# ---------------------------------------------------------------------

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# ---------------------------------------------------------------------
# 3. Convert categorical columns to numerical values
# ---------------------------------------------------------------------

df_encoded = pd.get_dummies(
    df,
    columns=["Sex", "Embarked"],
    drop_first=True
)

# ---------------------------------------------------------------------
# 4. Select features and target
# ---------------------------------------------------------------------

DROP_COLS = ["Survived", "Name", "Ticket", "Cabin"]

X = df_encoded.drop(columns=DROP_COLS)
y = df_encoded[TARGET]

feature_names = X.columns.tolist()

# ---------------------------------------------------------------------
# 5. Train / test split
# ---------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# ---------------------------------------------------------------------
# 6. Train a Random Forest
# ---------------------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    oob_score=True,
    random_state=RANDOM_STATE
)

rf.fit(X_train, y_train)

# ---------------------------------------------------------------------
# 7. Evaluate
# ---------------------------------------------------------------------

y_pred = rf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"\nOOB score: {rf.oob_score_:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Survived", "Survived"]
))

# ---------------------------------------------------------------------
# 8. Feature importances
# ---------------------------------------------------------------------

print("Feature importances:")

for name, importance in sorted(
    zip(feature_names, rf.feature_importances_),
    key=lambda x: -x[1]
):
    print(f"  {name}: {importance:.4f}")

Sample of the dataset:
   PassengerId  Survived  Pclass  \
0          892         0       3   
1          893         1       3   
2          894         0       2   
3          895         0       3   
4          896         1       3   

                                           Name     Sex   Age  SibSp  Parch  \
0                              Kelly, Mr. James    male  34.5      0      0   
1              Wilkes, Mrs. James (Ellen Needs)  female  47.0      1      0   
2                     Myles, Mr. Thomas Francis    male  62.0      0      0   
3                              Wirz, Mr. Albert    male  27.0      0      0   
4  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female  22.0      1      1   

    Ticket     Fare Cabin Embarked  
0   330911   7.8292   NaN        Q  
1   363272   7.0000   NaN        S  
2   240276   9.6875   NaN        Q  
3   315154   8.6625   NaN        S  
4  3101298  12.2875   NaN        S  

Dataset shape: (418, 12)

Class balance:
Survived
0    266
1  